In [1]:
import geemap
import ee
geemap.ee_initialize()
ee.Authenticate()

True

In [3]:
import ee
import geemap

# -----------------------------
# 1) Initialize Earth Engine
# -----------------------------
try:
    ee.Initialize()
except Exception:
    geemap.ee_initialize()
    ee.Initialize()

# -----------------------------
# 2) ROI (Haor bbox)
# -----------------------------
BBOX_BD = [89.8, 23.5, 92.8, 25.5]  # [west, south, east, north]
#BBOX_BD = [115.8, 28.4, 116.8, 29.8] poyang lake china
#BBOX_BD = [-60.1, -22.5, -54.7, -15.5] #the pantanal brazil
#BBOX_BD = [90.9, 24.9, 91.5, 25.3] # [west, south, east, north]


roi = ee.Geometry.Rectangle(BBOX_BD)

# -----------------------------
# 3) Dates (edit)
# -----------------------------
REF_START   = "2025-05-01"
REF_END     = "2025-05-15"
FLOOD_START = "2025-06-01"
FLOOD_END   = "2025-06-15"

# Orbit pass: None first (do NOT filter until you confirm data exists)
ORBIT_PASS = None  # "ASCENDING" / "DESCENDING" / None

# Main polarization to try first
POL_PRIMARY = "VV"   # then fallback to "VH" automatically if needed

# Flood threshold (dB): flood tends to be darker => negative delta
DELTA_DB_THR = -3.0

# Optional slope mask (reduces terrain artifacts; harmless in flat Bangladesh)
SLOPE_MAX_DEG = 5

# Optional cleanup
CLEAN_RADIUS_M = 60  # 30–100m typical

# -----------------------------
# 4) Helpers
# -----------------------------
def count_ic(ic, label):
    n = ic.size().getInfo()
    print(f"[DIAG] {label}: {n}")
    return n

def hist_prop(ic, prop, label):
    # Beware: getInfo can be heavy if you do too much; keep it minimal
    h = ic.aggregate_histogram(prop).getInfo()
    print(f"[DIAG] {label} histogram of '{prop}': {h}")

def base_s1(roi, start, end):
    # COPERNICUS/S1_GRD in GEE is already in dB (log scaled). No to_db needed.
    return (ee.ImageCollection("COPERNICUS/S1_GRD")
            .filterBounds(roi)
            .filterDate(start, end))

def filter_mode(ic, mode):
    return ic.filter(ee.Filter.eq("instrumentMode", mode))

def filter_pol(ic, pol):
    return (ic
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", pol))
            .select(pol))

def filter_orbit(ic, orbit_pass):
    if orbit_pass is None:
        return ic
    return ic.filter(ee.Filter.eq("orbitProperties_pass", orbit_pass))

def mask_border_noise(db_img):
    # mask very low values (border noise)
    return db_img.updateMask(db_img.gt(-30))

def safe_median_singleband(ic, band_name):
    """
    Always return a 1-band image named band_name.
    If ic is empty, return a fully masked constant band (so math won't crash).
    """
    empty = ee.Image.constant(0).rename(band_name).updateMask(ee.Image.constant(0))
    return ee.Image(ee.Algorithms.If(ic.size().gt(0),
                                     ic.median().rename(band_name),
                                     empty))

def pick_mode_and_pol(roi, ref_start, ref_end, flood_start, flood_end, orbit_pass, pol_primary):
    """
    Try modes in order: IW -> EW -> (no mode filter)
    Try pol: primary -> VH -> VV (if primary wasn't VV)
    Returns: (pol, mode_or_None, ref_ic, flood_ic)
    """
    modes = ["IW", "EW", None]
    pol_try = [pol_primary]
    if pol_primary != "VH":
        pol_try.append("VH")
    if pol_primary != "VV":
        pol_try.append("VV")

    # Use a slightly wider diagnostic window to detect availability
    diag_start = min(ref_start, flood_start)
    diag_end   = max(ref_end, flood_end)

    raw = base_s1(roi, diag_start, diag_end)
    n_raw = count_ic(raw, f"S1_GRD raw in ROI ({diag_start} to {diag_end})")
    if n_raw == 0:
        print("[ERROR] No Sentinel-1 GRD found in ROI at all. Double-check ROI coordinates.")
        return None, None, None, None

    # Optional hist to see what exists
    hist_prop(raw, "instrumentMode", "Raw")
    # transmitterReceiverPolarisation is a STRING_LIST so histogram can be noisy, but still useful
    hist_prop(raw, "orbitProperties_pass", "Raw")

    for mode in modes:
        ic_mode = raw if mode is None else filter_mode(raw, mode)
        n_mode = count_ic(ic_mode, f"After instrumentMode={mode}")
        if n_mode == 0:
            continue

        for pol in pol_try:
            ic_pol = filter_pol(ic_mode, pol)
            ic_pol = filter_orbit(ic_pol, orbit_pass)
            n_pol = count_ic(ic_pol, f"After POL={pol}, ORBIT={orbit_pass}")

            if n_pol == 0:
                continue

            # Build ref/flood subsets (same filtered collection)
            ref_ic = ic_pol.filterDate(ref_start, ref_end).map(lambda img: mask_border_noise(img).clip(roi))
            flood_ic = ic_pol.filterDate(flood_start, flood_end).map(lambda img: mask_border_noise(img).clip(roi))

            ref_n = count_ic(ref_ic, f"REF images ({ref_start} to {ref_end})")
            flood_n = count_ic(flood_ic, f"FLOOD images ({flood_start} to {flood_end})")

            if ref_n > 0 and flood_n > 0:
                print(f"[INFO] Using mode={mode}, pol={pol}, orbit={orbit_pass}")
                return pol, mode, ref_ic, flood_ic

    print("[ERROR] Could not find any combination that yields images in BOTH ref and flood windows.")
    print("        Fix: widen date ranges, set ORBIT_PASS=None, or accept EW mode if IW is empty.")
    return None, None, None, None

# -----------------------------
# 5) Choose data subset (robust)
# -----------------------------
pol, mode, ref_ic, flood_ic = pick_mode_and_pol(
    roi=roi,
    ref_start=REF_START, ref_end=REF_END,
    flood_start=FLOOD_START, flood_end=FLOOD_END,
    orbit_pass=ORBIT_PASS,
    pol_primary=POL_PRIMARY
)

# Stop if nothing found
if ref_ic is None:
    raise RuntimeError("No usable Sentinel-1 data found for your ROI/time windows. See diagnostics above.")

# -----------------------------
# 6) Composites + delta + flood mask
# -----------------------------
ref   = safe_median_singleband(ref_ic,   "ref_db")
flood = safe_median_singleband(flood_ic, "flood_db")
delta = flood.subtract(ref).rename("delta_db")

flood_mask = delta.lt(DELTA_DB_THR).rename("flood_mask")

# Slope mask (optional)
slope = ee.Terrain.slope(ee.Image("USGS/SRTMGL1_003"))
flood_mask = flood_mask.updateMask(slope.lt(SLOPE_MAX_DEG))

# Cleanup (optional)
flood_mask_clean = flood_mask.focal_mode(radius=CLEAN_RADIUS_M, units="meters").rename("flood_mask_clean")

# -----------------------------
# 7) Visualize
# -----------------------------
m = geemap.Map()
m.centerObject(roi, 7)

m.addLayer(ref,   {"min": -25, "max": 0}, f"REF {pol} dB ({REF_START} to {REF_END})", False)
m.addLayer(flood, {"min": -25, "max": 0}, f"FLOOD {pol} dB ({FLOOD_START} to {FLOOD_END})", True)
m.addLayer(delta, {"min": -8, "max": 2},  f"Delta dB (flood-ref), thr<{DELTA_DB_THR}", True)

m.addLayer(flood_mask.selfMask(),       {"palette": ["00FFFF"]}, "Flood mask (raw)", False)
m.addLayer(flood_mask_clean.selfMask(), {"palette": ["FF0000"]}, "Flood mask (clean)", True)

m.addLayer(ee.Image().paint(roi, 1, 2), {"palette": ["FFFFFF"]}, "ROI", True)

m


[DIAG] S1_GRD raw in ROI (2025-05-01 to 2025-06-15): 44
[DIAG] Raw histogram of 'instrumentMode': {'IW': 44}
[DIAG] Raw histogram of 'orbitProperties_pass': {'ASCENDING': 21, 'DESCENDING': 23}
[DIAG] After instrumentMode=IW: 44
[DIAG] After POL=VV, ORBIT=None: 44
[DIAG] REF images (2025-05-01 to 2025-05-15): 16
[DIAG] FLOOD images (2025-06-01 to 2025-06-15): 12
[INFO] Using mode=IW, pol=VV, orbit=None


Map(center=[24.504754997740452, 91.30000000000003], controls=(WidgetControl(options=['position', 'transparent_…